In [1]:
#!pip install torchvision

In [5]:
import torch
import os
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image

In [6]:
#load image => transform => dataset of all images 
class ImageProcessor:
    def __init__(self, root_dir_path, transformations = None):
        self.root_dir_path = root_dir_path
        self.transformations = transformations

        #list of path for all images
        self.all_img_paths = [os.path.join(root_dir_path, img) for img in os.listdir(root_dir_path)]

    def __len__(self):
        return len(self.all_img_paths)

    def __getitem__(self, idx):
        img_path = self.all_img_paths[idx]
        img = Image.open(img_path).convert("RGB")

        if self.transformations:
            img = self.transformations(img)

        return img

In [8]:
root_dir_path = "./img_align_celeba"

transformations = transforms.Compose([
    transforms.CenterCrop(178),
    transforms.Resize(64),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [9]:
dataset = ImageProcessor(root_dir_path, transformations)
print(f"loaded {len(dataset)} images")

FileNotFoundError: [WinError 3] The system cannot find the path specified: './img_align_celeba'

In [ ]:
dataloader = DataLoader(dataset, batch_size = 128, shuffle = True)

GENERATOR NETWORK

In [ ]:
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [ ]:
class Generator(nn.Module):
    def __init__(self, z_dim = 100, img_channels = 3):
        super(Generator, self).__init__()

        #fully connected layer
        self.model = nn.Sequential(
            nn.Linear(z_dim, 256),
            nn.ReLU(),
            
            nn.Linear(256, 512),
            nn.ReLU(),

            nn.Linear(512, 1024),
            nn.ReLU(),

            nn.Linear(1024, 64 * 64 * img_channels),
            nn.Tanh()      #[-1, 1]
        )

    def forward(self, z):
        img = self.model(z)
        img = img.view(img.size(0), 3, 64, 64)
        return img

        #fake img => 64 x 64 x 3 x batch_size   (kyuki hum generator ko batch provide kr rahe hai) [IT IS A 4D TENSOR]

DISCRIMINATOR NETWORK

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, z_dim = 100, img_channels = 3):
        super(Discriminator, self).__init__()

        #fully connected layer
        self.model = nn.Sequential(
            nn.Flatten(),    #convert from 4D Tensor to 1D Tensor
            
            nn.Linear(img_channels * 64 * 64, 1024),    #agar DCGAN hota, toh yaha pe convolutional layers hote
            nn.LeakyReLU(0.2, inplace = True),          #to prevent vanishing gradient

            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2, inplace = True),          #to prevent vanishing gradient

            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace = True),          #to prevent vanishing gradient

            nn.Linear(256, 1),
            nn.Sigmoid()      #probability of true/fake
        )

    def forward(self, z):
        return self.model(z)

In [ ]:
GAN_loss = nn.BCELoss()

generator = Generator()
g_optimizer = optim.Adam(generator.parameters(), lr = 0.0002, betas = (0.5, 0.999))

discriminator = Discriminator()
d_optimizer = optim.Adam(discriminator.parameters(), lr = 0.0002 , betas = (0.5, 0.999))

In [ ]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

In [ ]:
generator = generator.to(device)
discriminator = discriminator.to(device)

In [1]:
def train(generator, discriminator, dataloader, epochs=10):
    for epoch in range(epochs):
        for i, imgs in enumerate(dataloader):
            real_imgs = imgs.to(device)
            batch_size = real_imgs.size(0)

            #create real img labels and fake img labels
            real_labels = torch.ones(batch_size, 1).to(device)           # [1, 1, 1, 1 ......]
            fake_labels = torch.zeros(batch_size, 1).to(device)          # [0, 0, 0, 0 ......]

            #Train the discriminator
            d_optimizer.zero_grad()

            fake_imgs = generator(torch.randn(batch_size, 100).to(device))    #input that we pass to generator must also be on the same device

            real_loss = GAN_loss(discriminator(real_imgs), real_labels)
            fake_loss = GAN_loss(discriminator(fake_imgs.detach()), fake_labels)

            d_loss = (real_loss + fake_loss)/2

            d_loss.backward()
            d_optimizer.step()

            g_optimizer.zero_grad()

            g_loss = GAN_loss(discriminator(fake_imgs), real_labels)

            g_loss.backward()
            g_optimizer.step()

            if i % 50 == 0:
                print(f"for epoch: {epoch+1}/{epochs} .... batch: {i+1}.... G loss: {g_loss} ... D loss: {d_loss}")

        # save generated image for each epoch
        save_generated_images(generator, epoch, device)

_IncompleteInputError: incomplete input (3232574319.py, line 2)

In [ ]:
import matplotlib.pyplot as plt
import torchvision

def save_generated_images(generator, epoch, device, num_imgs = 8):
    z = torch.randn(num_imgs, 100).to(device)
    generated_imgs = generator(z).detach().cpu()

    """Generator creates image that are [-1, 1], but for RGB we need [0, 1] , se we put normalize = True"""
    grid = torchvision.util.make_grid(generated_imgs, nrow = 4, normalize = True)

    plt.imshow(np.transpose(grid, (1, 2, 0))) 
    #grid tensor gives : channel x H x W (3 x 64 x 64) but matplotlib wants H x W x Channels 
    plt.title(f"epoch {epoch+1}")
    plt.axis("off")
    plt.show()

In [ ]:
train(generator, discriminator, dataloader, epoch = 5)